# RVG Unified Field - Multi-Drone Swarm Simulation

Interactive demonstration of multi-drone swarm simulations using PyBullet physics engine with **Refractive Vacuum Gravity (RVG) Unified Field** propulsion.

## Features

- **RVG Propulsion Physics**: Master Equation of Levitation thrust calculations
- **Dilaton Enhancement**: Non-linear Θ_dilaton(B) vacuum response modeling
- **MADA Integration**: 200-500x magnetic amplification per U.S. Patent 5,929,732
- **Supra-Saturation Regime**: Material-specific B/B_sat optimization
- **Swarm Behaviors**: Formation flight, asymmetric warfare scenarios
- **3D Visualization**: Trajectory plotting with RVG telemetry
- **Scalable**: From 5 to 50+ drones with different material configurations

## Framework References

- [RVG Unified Field Theory](https://dx.doi.org/10.2139/ssrn.5381654) (Hofseth, 2025)
- [U.S. Patent #5,929,732 - MADA](https://patents.google.com/patent/US5929732A/en)
- CMS/ATLAS 95.4 GeV di-photon resonance (3.1σ combined significance)

## Key Equations

**Master Equation of Levitation:**
$$\mathbf{F}_{\text{lift}} = \int_V \frac{1}{2\mu_0} \Theta_{\text{dilaton}}(B) \cdot \nabla B^2 \, dV$$

**Dilaton Enhancement:**
$$\Theta_{\text{dilaton}}(B) = \theta_{\text{base}} \cdot (1 + (B/B_{\text{crit}})^2) \cdot f_{\text{activation}}(B)$$

## Requirements

```bash
pip install pybullet numpy matplotlib
```

---

## 1. Import Dependencies and RVG Framework

In [ ]:
import sys
import os
import numpy as np
import time
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

# PyBullet
try:
    import pybullet as p
    import pybullet_data
    PYBULLET_AVAILABLE = True
    print("✓ PyBullet loaded successfully")
except ImportError:
    PYBULLET_AVAILABLE = False
    print("✗ PyBullet not available. Install with: pip install pybullet")
    print("  Continuing with simplified simulation...")

# Add parent directory to path
notebook_dir = os.getcwd()
parent_dir = os.path.dirname(notebook_dir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Import project modules
MODULES_AVAILABLE = False
try:
    from simulations.thrust_model import calculate_thrust_params
    from simulations.equations import force_vector, total_thrust, acceleration
    MODULES_AVAILABLE = True
    print("✓ QED modules loaded successfully")
except ImportError as e:
    print(f"Note: Using standalone RVG functions (project modules not found)")

print("\n" + "="*70)
print("RVG UNIFIED FIELD - SWARM PROPULSION SIMULATION")
print("="*70)

## 2. RVG Framework Constants and Functions

In [ ]:
# =============================================================================
# RVG UNIFIED FIELD CONSTANTS
# =============================================================================

# Fundamental constants
MU_0 = 4 * np.pi * 1e-7  # Vacuum permeability (H/m)
EPSILON_0 = 8.854187817e-12  # Vacuum permittivity (F/m)
C = 299792458.0  # Speed of light (m/s)

# 95 GeV Dilaton/Radion Resonance Parameters
DILATON_MASS = 95.4  # GeV - observed CMS/ATLAS resonance
DILATON_SIGMA = 3.1  # Combined significance (σ)

# Default RVG parameters (require experimental calibration)
DEFAULT_THETA_BASE = 1e-6  # Base dilaton enhancement (placeholder)
DEFAULT_B_CRIT = 20.0  # Effective critical field for activation (T)
DEFAULT_GAMMA = 0.1  # Activation steepness
DEFAULT_EPSILON = 0.01  # Activation offset

# Material saturation limits (from Materials Ranking)
MATERIALS = {
    'Minnealloy': {'B_sat': 2.85, 'score': 95, 'color': [0.2, 0.8, 0.2, 1.0]},  # Green - BEST
    'Hiperco-50': {'B_sat': 2.40, 'score': 88, 'color': [0.2, 0.2, 0.8, 1.0]},  # Blue
    'Pure_Iron': {'B_sat': 2.10, 'score': 90, 'color': [0.8, 0.5, 0.2, 1.0]},   # Orange
    'Finemet': {'B_sat': 1.20, 'score': 96, 'color': [0.8, 0.2, 0.8, 1.0]}      # Purple
}

# MADA Amplification (per U.S. Patent 5,929,732)
MADA_K_DEFAULT = 200.0  # Default amplification (~200x vs single magnet)
MADA_K_MAX = 529.0  # Maximum theoretical amplification

# Default system parameters
DEFAULT_VOLUME = 0.1  # Integration volume (m³)
DEFAULT_ETA = 0.95  # Alignment efficiency

print("RVG Framework Constants Loaded:")
print(f"  Dilaton mass: {DILATON_MASS} GeV ({DILATON_SIGMA}σ significance)")
print(f"  Default θ_base: {DEFAULT_THETA_BASE}")
print(f"  Default B_crit: {DEFAULT_B_CRIT} T")
print(f"  MADA amplification: {MADA_K_DEFAULT}-{MADA_K_MAX}x")
print(f"\nMaterials available:")
for name, props in MATERIALS.items():
    print(f"  {name}: B_sat={props['B_sat']}T, score={props['score']}/100")

In [ ]:
# =============================================================================
# RVG CORE PROPULSION FUNCTIONS
# =============================================================================

def theta_dilaton(B, theta_base=DEFAULT_THETA_BASE, B_crit=DEFAULT_B_CRIT,
                  gamma=DEFAULT_GAMMA, epsilon=DEFAULT_EPSILON):
    """
    Calculate dilaton enhancement factor Θ_dilaton(B).
    
    Θ_dilaton(B) = θ_base * (1 + (B/B_crit)²) * exp(-γ/(B/B_crit + ε))
    
    The enhancement represents the non-linear vacuum response due to
    the 95 GeV resonance trace anomaly coupling.
    """
    ratio = B / (B_crit + 1e-10)
    polynomial = 1.0 + ratio**2
    activation = np.exp(-gamma / (ratio + epsilon))
    return theta_base * polynomial * activation


def vacuum_refractive_index(B, theta_base=DEFAULT_THETA_BASE, B_crit=DEFAULT_B_CRIT):
    """
    Calculate vacuum refractive index K(r).
    
    K = 1 + Θ_dilaton(B) * (B/B_crit)²
    """
    theta = theta_dilaton(B, theta_base, B_crit)
    chi_vac = theta * (B / B_crit)**2
    return 1.0 + chi_vac


def supra_saturation_effectiveness(B_opposing, B_sat, n=2.0):
    """
    Calculate supra-saturation effectiveness.
    
    Effects manifest when B_opposing >> B_sat (driving μ_eff ≈ 1).
    Returns effectiveness factor 0-1.
    """
    ratio = B_opposing / (B_sat + 1e-10)
    if ratio > 5.0:
        return 1.0  # Full effectiveness in deep supra-saturation
    elif ratio > 2.0:
        return 0.3 + 0.7 * (ratio - 2.0) / 3.0  # Linear ramp
    elif ratio > 1.0:
        return 0.1 + 0.2 * (ratio - 1.0)  # Near saturation
    else:
        return 0.1 * ratio  # Sub-saturation (weak)


def mada_amplification(B_source, k=MADA_K_DEFAULT):
    """
    Apply MADA amplification to source B-field.
    
    Per U.S. Patent 5,929,732, MADA enables ~200-500x effective
    amplification through magnetic focusing/frustration.
    """
    return B_source * k / 216.0  # Normalized to 6x distance equivalent


def master_equation_thrust(B, grad_B2, volume, material='Minnealloy',
                           theta_base=DEFAULT_THETA_BASE, B_crit=DEFAULT_B_CRIT,
                           eta=DEFAULT_ETA, mada_k=MADA_K_DEFAULT):
    """
    Master Equation of Levitation thrust calculation.
    
    F_lift = (1/2μ₀) * Θ_dilaton(B) * ∇(B²) * V * η * effectiveness
    
    Parameters:
    - B: Operating magnetic field (T)
    - grad_B2: Gradient of B² (T²/m)
    - volume: Integration volume (m³)
    - material: Core material for B_sat lookup
    - theta_base: Dilaton base enhancement
    - B_crit: Critical field for dilaton activation
    - eta: Alignment efficiency
    - mada_k: MADA amplification factor
    
    Returns:
    - thrust: Force magnitude (N)
    - theta: Dilaton enhancement factor
    - effectiveness: Supra-saturation effectiveness
    """
    # Apply MADA amplification
    B_effective = mada_amplification(B, mada_k)
    
    # Get material properties
    B_sat = MATERIALS.get(material, MATERIALS['Minnealloy'])['B_sat']
    
    # Calculate components
    theta = theta_dilaton(B_effective, theta_base, B_crit)
    effectiveness = supra_saturation_effectiveness(B_effective, B_sat)
    
    # Master Equation thrust
    F = (1.0 / (2.0 * MU_0)) * theta * grad_B2 * volume * eta * effectiveness
    
    return F, theta, effectiveness


print("\nRVG Propulsion Functions Defined:")
print("  - theta_dilaton(B, θ_base, B_crit, γ, ε)")
print("  - vacuum_refractive_index(B, θ_base, B_crit)")
print("  - supra_saturation_effectiveness(B_opposing, B_sat)")
print("  - mada_amplification(B_source, k)")
print("  - master_equation_thrust(B, ∇B², V, material, ...)")

## 3. Simulation Configuration

Configure swarm parameters and RVG propulsion settings:

In [ ]:
# =============================================================================
# SIMULATION CONFIGURATION
# =============================================================================

# Swarm parameters
NUM_DRONES = 12  # Number of drones in swarm
DRONE_MASS = 20000.0  # kg per drone
SIM_TIME = 30.0  # Simulation duration (seconds)
TIME_STEP = 1/240  # Physics timestep
REAL_TIME = False  # False = faster simulation
GUI_MODE = True if PYBULLET_AVAILABLE else False  # PyBullet GUI

# Scenario selection
# Options: 'asymmetric', 'formation', 'material_comparison', 'mada_test'
SCENARIO = 'material_comparison'

# RVG Propulsion parameters
B_SOURCE = 3.0  # Source B-field per MADA unit (T) - e.g., N52 magnet stack
MADA_K = 200.0  # MADA amplification factor
GRAD_B2_BASE = 1e10  # Base ∇B² gradient (T²/m)
VOLUME = 0.1  # Integration volume per drone (m³)
N_UNITS = 24  # Number of MADA units per drone

# Pulsing parameters
PULSE_FREQ = 100.0  # Hz (50-1000 Hz range)
DUTY_CYCLE = 0.5  # 50% duty cycle

# Visualization
RECORD_INTERVAL = 10  # Record position every N steps

print(f"Swarm Configuration:")
print(f"  Drones: {NUM_DRONES}")
print(f"  Mass per drone: {DRONE_MASS} kg")
print(f"  Scenario: {SCENARIO}")
print(f"  Duration: {SIM_TIME}s")
print(f"\nRVG Propulsion:")
print(f"  B_source: {B_SOURCE} T")
print(f"  MADA amplification: {MADA_K}x")
print(f"  B_effective: {mada_amplification(B_SOURCE, MADA_K):.1f} T")
print(f"  ∇B² (base): {GRAD_B2_BASE:.1e} T²/m")
print(f"  N_units: {N_UNITS}")
print(f"  Pulse frequency: {PULSE_FREQ} Hz")
print(f"\nPhysics:")
print(f"  GUI: {'Enabled' if GUI_MODE else 'Disabled'}")
print(f"  Real-time: {REAL_TIME}")

## 4. Calculate Thrust for Each Drone Configuration

In [ ]:
# =============================================================================
# DRONE THRUST CALCULATIONS
# =============================================================================

# Store drone configurations
drone_configs = []

# Calculate base thrust using Master Equation
B_eff = mada_amplification(B_SOURCE, MADA_K)
print(f"\nBase Thrust Calculation (Master Equation of Levitation):")
print(f"  B_source = {B_SOURCE} T")
print(f"  B_effective (with MADA) = {B_eff:.1f} T")
print(f"  ∇B² = {GRAD_B2_BASE:.1e} T²/m")
print(f"  Volume = {VOLUME} m³")

# Reference thrust with Minnealloy
T_ref, theta_ref, eff_ref = master_equation_thrust(
    B_SOURCE, GRAD_B2_BASE, VOLUME, 'Minnealloy',
    mada_k=MADA_K
)
T_ref *= N_UNITS  # Scale by number of MADA units
a_ref = T_ref / DRONE_MASS

print(f"\nReference drone (Minnealloy):")
print(f"  Θ_dilaton = {theta_ref:.2e}")
print(f"  Supra-sat effectiveness = {eff_ref:.2f}")
print(f"  Thrust = {T_ref:.0f} N ({T_ref/1000:.1f} kN)")
print(f"  Acceleration = {a_ref:.2f} m/s² ({a_ref/9.81:.2f}g)")

# Configure drones based on scenario
if SCENARIO == 'asymmetric':
    # Advanced drones (Minnealloy, high MADA) vs Standard (Iron, low MADA)
    print(f"\nAsymmetric Warfare Scenario:")
    for i in range(NUM_DRONES):
        if i < NUM_DRONES // 2:
            # Advanced drones
            material = 'Minnealloy'
            mada = MADA_K * 1.5
            role = 'Advanced'
        else:
            # Standard drones
            material = 'Pure_Iron'
            mada = MADA_K * 0.5
            role = 'Standard'
        
        T, theta, eff = master_equation_thrust(
            B_SOURCE, GRAD_B2_BASE, VOLUME, material, mada_k=mada
        )
        T *= N_UNITS
        
        drone_configs.append({
            'id': i,
            'material': material,
            'mada_k': mada,
            'thrust': T,
            'theta': theta,
            'effectiveness': eff,
            'role': role,
            'color': MATERIALS[material]['color']
        })
    
    adv_thrust = np.mean([d['thrust'] for d in drone_configs if d['role'] == 'Advanced'])
    std_thrust = np.mean([d['thrust'] for d in drone_configs if d['role'] == 'Standard'])
    print(f"  Advanced ({NUM_DRONES//2}): {adv_thrust:.0f} N avg")
    print(f"  Standard ({NUM_DRONES - NUM_DRONES//2}): {std_thrust:.0f} N avg")
    print(f"  Thrust ratio: {adv_thrust/std_thrust:.1f}x")

elif SCENARIO == 'material_comparison':
    # Each drone uses a different material
    print(f"\nMaterial Comparison Scenario:")
    material_list = list(MATERIALS.keys())
    
    for i in range(NUM_DRONES):
        material = material_list[i % len(material_list)]
        
        T, theta, eff = master_equation_thrust(
            B_SOURCE, GRAD_B2_BASE, VOLUME, material, mada_k=MADA_K
        )
        T *= N_UNITS
        
        drone_configs.append({
            'id': i,
            'material': material,
            'mada_k': MADA_K,
            'thrust': T,
            'theta': theta,
            'effectiveness': eff,
            'role': material,
            'color': MATERIALS[material]['color']
        })
    
    print("  Drone configurations:")
    for mat in material_list:
        mat_drones = [d for d in drone_configs if d['material'] == mat]
        if mat_drones:
            avg_thrust = np.mean([d['thrust'] for d in mat_drones])
            avg_eff = np.mean([d['effectiveness'] for d in mat_drones])
            print(f"    {mat}: {len(mat_drones)} drones, {avg_thrust:.0f} N, eff={avg_eff:.2f}")

elif SCENARIO == 'formation':
    # Leader (high power) + followers
    print(f"\nFormation Flight Scenario:")
    for i in range(NUM_DRONES):
        if i == 0:
            # Leader - maximum configuration
            material = 'Minnealloy'
            mada = MADA_K_MAX
            role = 'Leader'
        else:
            # Followers - standard configuration
            material = 'Minnealloy'
            mada = MADA_K
            role = 'Follower'
        
        T, theta, eff = master_equation_thrust(
            B_SOURCE, GRAD_B2_BASE, VOLUME, material, mada_k=mada
        )
        T *= N_UNITS
        
        color = [0.9, 0.9, 0.0, 1.0] if role == 'Leader' else MATERIALS[material]['color']
        
        drone_configs.append({
            'id': i,
            'material': material,
            'mada_k': mada,
            'thrust': T,
            'theta': theta,
            'effectiveness': eff,
            'role': role,
            'color': color
        })
    
    leader_T = drone_configs[0]['thrust']
    follower_T = np.mean([d['thrust'] for d in drone_configs[1:]])
    print(f"  Leader thrust: {leader_T:.0f} N")
    print(f"  Follower thrust (avg): {follower_T:.0f} N")

elif SCENARIO == 'mada_test':
    # Varying MADA amplification
    print(f"\nMADA Amplification Test Scenario:")
    mada_values = np.linspace(100, 500, NUM_DRONES)
    
    for i, mada in enumerate(mada_values):
        T, theta, eff = master_equation_thrust(
            B_SOURCE, GRAD_B2_BASE, VOLUME, 'Minnealloy', mada_k=mada
        )
        T *= N_UNITS
        
        # Color gradient based on MADA
        color_val = (mada - 100) / 400
        color = [color_val, 0.5, 1-color_val, 1.0]
        
        drone_configs.append({
            'id': i,
            'material': 'Minnealloy',
            'mada_k': mada,
            'thrust': T,
            'theta': theta,
            'effectiveness': eff,
            'role': f'MADA-{mada:.0f}',
            'color': color
        })
    
    print(f"  MADA range: {mada_values[0]:.0f}x to {mada_values[-1]:.0f}x")
    print(f"  Thrust range: {drone_configs[0]['thrust']:.0f} to {drone_configs[-1]['thrust']:.0f} N")

else:
    # Default: all same configuration
    for i in range(NUM_DRONES):
        drone_configs.append({
            'id': i,
            'material': 'Minnealloy',
            'mada_k': MADA_K,
            'thrust': T_ref,
            'theta': theta_ref,
            'effectiveness': eff_ref,
            'role': 'Standard',
            'color': MATERIALS['Minnealloy']['color']
        })

print(f"\n✓ Configured {NUM_DRONES} drones")
print(f"  Thrust range: {min([d['thrust'] for d in drone_configs]):.0f} - {max([d['thrust'] for d in drone_configs]):.0f} N")

## 5. Initialize Physics Engine and Spawn Drones

In [ ]:
# =============================================================================
# PHYSICS INITIALIZATION
# =============================================================================

# Storage for simulation data
drone_ids = []
trajectories = [[] for _ in range(NUM_DRONES)]
telemetry = {
    'time': [],
    'positions': [[] for _ in range(NUM_DRONES)],
    'velocities': [[] for _ in range(NUM_DRONES)],
    'thrusts': [[] for _ in range(NUM_DRONES)],
    'theta_values': [[] for _ in range(NUM_DRONES)]
}

if PYBULLET_AVAILABLE:
    # Connect to PyBullet
    if GUI_MODE:
        physicsClient = p.connect(p.GUI)
        print("PyBullet GUI started")
    else:
        physicsClient = p.connect(p.DIRECT)
        print("PyBullet running in headless mode")
    
    # Configure physics
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.81)
    p.setTimeStep(TIME_STEP)
    p.setRealTimeSimulation(0)
    
    # Load ground plane
    planeId = p.loadURDF("plane.urdf")
    
    # Camera setup
    if GUI_MODE:
        p.resetDebugVisualizerCamera(
            cameraDistance=30,
            cameraYaw=45,
            cameraPitch=-30,
            cameraTargetPosition=[0, 0, 10]
        )
    
    print(f"✓ Physics engine initialized")
else:
    print("Running simplified simulation (no PyBullet)")

# Generate starting positions
start_positions = []
if SCENARIO == 'formation':
    # V-formation
    for i in range(NUM_DRONES):
        x = 0 if i == 0 else (i - 1) * 3 * (1 if (i-1) % 2 == 0 else -1)
        y = 0 if i == 0 else -(i // 2) * 3
        z = 5.0
        start_positions.append([x, y, z])
else:
    # Grid formation
    grid_size = int(np.ceil(np.sqrt(NUM_DRONES)))
    for i in range(NUM_DRONES):
        x = (i % grid_size - grid_size / 2) * 5.0
        y = (i // grid_size - grid_size / 2) * 5.0
        z = 5.0
        start_positions.append([x, y, z])

# Spawn drones
if PYBULLET_AVAILABLE:
    for i, (pos, config) in enumerate(zip(start_positions, drone_configs)):
        drone_id = p.loadURDF(
            "sphere2.urdf",
            pos,
            globalScaling=0.8
        )
        
        p.changeDynamics(
            drone_id, -1,
            mass=DRONE_MASS,
            linearDamping=0.05,
            angularDamping=0.05
        )
        
        p.changeVisualShape(
            drone_id, -1,
            rgbaColor=config['color']
        )
        
        drone_ids.append(drone_id)
        trajectories[i].append(np.array(pos))

print(f"✓ Spawned {NUM_DRONES} drones")

## 6. Run Physics Simulation with RVG Propulsion

In [ ]:
# =============================================================================
# MAIN SIMULATION LOOP
# =============================================================================

print(f"\nStarting RVG Swarm Simulation...")
print(f"Duration: {SIM_TIME}s, Steps: {int(SIM_TIME/TIME_STEP)}")
print("="*60)

total_steps = int(SIM_TIME / TIME_STEP)
record_counter = 0
pulse_counter = 0
pulse_period = int(1.0 / (PULSE_FREQ * TIME_STEP))

start_time = time.time()

if PYBULLET_AVAILABLE:
    for step in range(total_steps):
        sim_time = step * TIME_STEP
        
        # Pulsing modulation
        pulse_phase = (step % pulse_period) / pulse_period
        pulse_active = pulse_phase < DUTY_CYCLE
        pulse_factor = 1.0 if pulse_active else 0.2
        
        # Apply thrust to each drone
        for i, (drone_id, config) in enumerate(zip(drone_ids, drone_configs)):
            # Base thrust from Master Equation
            thrust = config['thrust'] * pulse_factor
            
            # Get current position for directional control
            pos, orn = p.getBasePositionAndOrientation(drone_id)
            vel, ang_vel = p.getBaseVelocity(drone_id)
            
            # Simple altitude hold: increase thrust if below target
            target_altitude = 20.0
            altitude_error = target_altitude - pos[2]
            altitude_correction = np.clip(altitude_error * 5000, -thrust * 0.3, thrust * 0.3)
            
            # Apply thrust (vertical + correction)
            thrust_vec = [0, 0, thrust + altitude_correction]
            p.applyExternalForce(
                drone_id, -1,
                thrust_vec,
                [0, 0, 0],
                p.LINK_FRAME
            )
            
            # Formation keeping (attraction to neighbors)
            if SCENARIO in ['formation', 'material_comparison']:
                for j, other_id in enumerate(drone_ids):
                    if i != j:
                        other_pos, _ = p.getBasePositionAndOrientation(other_id)
                        diff = np.array(other_pos) - np.array(pos)
                        dist = np.linalg.norm(diff)
                        if 5.0 < dist < 15.0:  # Attraction range
                            attraction = diff / dist * 500
                            p.applyExternalForce(
                                drone_id, -1,
                                attraction.tolist(),
                                [0, 0, 0],
                                p.WORLD_FRAME
                            )
                        elif dist < 3.0:  # Collision avoidance
                            repulsion = -diff / (dist + 0.1) * 2000
                            p.applyExternalForce(
                                drone_id, -1,
                                repulsion.tolist(),
                                [0, 0, 0],
                                p.WORLD_FRAME
                            )
            
            # Record telemetry
            if record_counter % RECORD_INTERVAL == 0:
                trajectories[i].append(np.array(pos))
                telemetry['positions'][i].append(pos)
                telemetry['velocities'][i].append(vel)
                telemetry['thrusts'][i].append(thrust)
                telemetry['theta_values'][i].append(config['theta'])
        
        if record_counter % RECORD_INTERVAL == 0:
            telemetry['time'].append(sim_time)
        
        # Step simulation
        p.stepSimulation()
        record_counter += 1
        
        # Real-time sleep
        if REAL_TIME:
            time.sleep(TIME_STEP)
        
        # Progress
        if step % (total_steps // 10) == 0:
            progress = (step / total_steps) * 100
            avg_alt = np.mean([p.getBasePositionAndOrientation(d)[0][2] for d in drone_ids])
            print(f"  {progress:3.0f}% | t={sim_time:5.1f}s | avg_alt={avg_alt:5.1f}m")

else:
    # Simplified simulation without PyBullet
    positions = [np.array(pos) for pos in start_positions]
    velocities = [np.zeros(3) for _ in range(NUM_DRONES)]
    
    for step in range(total_steps):
        sim_time = step * TIME_STEP
        
        for i, config in enumerate(drone_configs):
            # Simple physics
            thrust = config['thrust']
            accel = np.array([0, 0, thrust / DRONE_MASS - 9.81])
            velocities[i] += accel * TIME_STEP
            positions[i] += velocities[i] * TIME_STEP
            positions[i][2] = max(0.1, positions[i][2])  # Ground collision
            
            if record_counter % RECORD_INTERVAL == 0:
                trajectories[i].append(positions[i].copy())
        
        record_counter += 1
        
        if step % (total_steps // 10) == 0:
            progress = (step / total_steps) * 100
            avg_alt = np.mean([p[2] for p in positions])
            print(f"  {progress:3.0f}% | t={sim_time:5.1f}s | avg_alt={avg_alt:5.1f}m")

elapsed_time = time.time() - start_time

print("="*60)
print(f"✓ Simulation complete")
print(f"  Real time: {elapsed_time:.2f}s")
print(f"  Sim time: {SIM_TIME:.1f}s")
print(f"  Speed: {SIM_TIME/elapsed_time:.1f}x real-time")

## 7. Collect Final Statistics

In [ ]:
# =============================================================================
# FINAL STATISTICS
# =============================================================================

final_positions = []
final_velocities = []
distances_traveled = []

if PYBULLET_AVAILABLE:
    for i, drone_id in enumerate(drone_ids):
        pos, _ = p.getBasePositionAndOrientation(drone_id)
        vel, _ = p.getBaseVelocity(drone_id)
        final_positions.append(pos)
        final_velocities.append(vel)
        
        traj = np.array(trajectories[i])
        if len(traj) > 1:
            dist = np.sum(np.linalg.norm(np.diff(traj, axis=0), axis=1))
            distances_traveled.append(dist)
else:
    final_positions = [np.array(trajectories[i][-1]) for i in range(NUM_DRONES)]
    final_velocities = [np.zeros(3) for _ in range(NUM_DRONES)]
    for i in range(NUM_DRONES):
        traj = np.array(trajectories[i])
        if len(traj) > 1:
            dist = np.sum(np.linalg.norm(np.diff(traj, axis=0), axis=1))
            distances_traveled.append(dist)

print("\nFinal Swarm Statistics:")
print("="*60)
print(f"  Average altitude: {np.mean([p[2] for p in final_positions]):.2f} m")
print(f"  Max altitude: {np.max([p[2] for p in final_positions]):.2f} m")
print(f"  Average speed: {np.mean([np.linalg.norm(v) for v in final_velocities]):.2f} m/s")
if distances_traveled:
    print(f"  Total distance (avg): {np.mean(distances_traveled):.2f} m")
print(f"  Swarm spread (X): {np.std([p[0] for p in final_positions]):.2f} m")
print(f"  Swarm spread (Y): {np.std([p[1] for p in final_positions]):.2f} m")

# Per-configuration stats
if SCENARIO == 'material_comparison':
    print(f"\nPer-Material Performance:")
    for mat in MATERIALS.keys():
        mat_indices = [i for i, c in enumerate(drone_configs) if c['material'] == mat]
        if mat_indices:
            mat_alts = [final_positions[i][2] for i in mat_indices]
            mat_thrust = np.mean([drone_configs[i]['thrust'] for i in mat_indices])
            print(f"  {mat}: avg_alt={np.mean(mat_alts):.1f}m, thrust={mat_thrust:.0f}N")

## 8. Disconnect Physics Engine

In [ ]:
if PYBULLET_AVAILABLE:
    p.disconnect()
    print("✓ PyBullet disconnected")

## 9. Visualize 3D Trajectories

In [ ]:
# =============================================================================
# 3D TRAJECTORY VISUALIZATION
# =============================================================================

fig = plt.figure(figsize=(16, 12))
ax = fig.add_subplot(111, projection='3d')

# Plot each drone's trajectory
for i, traj in enumerate(trajectories):
    traj_array = np.array(traj)
    if len(traj_array) > 1:
        config = drone_configs[i]
        color = config['color'][:3]
        label = f"{config['role']} ({config['material']})" if i < 4 else None
        
        ax.plot(
            traj_array[:, 0],
            traj_array[:, 1],
            traj_array[:, 2],
            color=color,
            linewidth=2,
            alpha=0.7,
            label=label
        )
        
        # Start marker
        ax.scatter(
            traj_array[0, 0], traj_array[0, 1], traj_array[0, 2],
            color=color, marker='o', s=100, edgecolors='black'
        )
        # End marker
        ax.scatter(
            traj_array[-1, 0], traj_array[-1, 1], traj_array[-1, 2],
            color=color, marker='s', s=100, edgecolors='black'
        )

ax.set_xlabel('X Position (m)', fontsize=12)
ax.set_ylabel('Y Position (m)', fontsize=12)
ax.set_zlabel('Z Position (m)', fontsize=12)
ax.set_title(
    f'RVG Swarm Simulation - {SCENARIO.replace("_", " ").title()} ({NUM_DRONES} Drones)',
    fontsize=14,
    fontweight='bold'
)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. RVG Telemetry Analysis

In [ ]:
# =============================================================================
# RVG TELEMETRY ANALYSIS
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Altitude over time
ax1 = axes[0, 0]
for i, traj in enumerate(trajectories):
    if len(traj) > 1:
        traj_array = np.array(traj)
        times = np.arange(len(traj)) * RECORD_INTERVAL * TIME_STEP
        ax1.plot(times, traj_array[:, 2], 
                 color=drone_configs[i]['color'][:3], alpha=0.7, linewidth=1.5)
ax1.set_xlabel('Time (s)', fontsize=11)
ax1.set_ylabel('Altitude (m)', fontsize=11)
ax1.set_title('Altitude vs Time', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# 2. Speed distribution
ax2 = axes[0, 1]
for i, traj in enumerate(trajectories):
    if len(traj) > 2:
        traj_array = np.array(traj)
        velocities = np.diff(traj_array, axis=0) / (RECORD_INTERVAL * TIME_STEP)
        speeds = np.linalg.norm(velocities, axis=1)
        times = np.arange(len(speeds)) * RECORD_INTERVAL * TIME_STEP
        ax2.plot(times, speeds, color=drone_configs[i]['color'][:3], alpha=0.7, linewidth=1.5)
ax2.set_xlabel('Time (s)', fontsize=11)
ax2.set_ylabel('Speed (m/s)', fontsize=11)
ax2.set_title('Speed vs Time', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 3. XY projection (top view)
ax3 = axes[0, 2]
for i, traj in enumerate(trajectories):
    if len(traj) > 1:
        traj_array = np.array(traj)
        ax3.plot(traj_array[:, 0], traj_array[:, 1], 
                 color=drone_configs[i]['color'][:3], alpha=0.7, linewidth=1.5)
        ax3.scatter(traj_array[-1, 0], traj_array[-1, 1], 
                   color=drone_configs[i]['color'][:3], s=50, edgecolors='black')
ax3.set_xlabel('X Position (m)', fontsize=11)
ax3.set_ylabel('Y Position (m)', fontsize=11)
ax3.set_title('Swarm Formation (Top View)', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.set_aspect('equal')

# 4. Thrust by material/config
ax4 = axes[1, 0]
thrusts = [c['thrust']/1000 for c in drone_configs]  # kN
colors = [c['color'][:3] for c in drone_configs]
labels = [c['role'] for c in drone_configs]
ax4.bar(range(NUM_DRONES), thrusts, color=colors, edgecolor='black')
ax4.set_xlabel('Drone ID', fontsize=11)
ax4.set_ylabel('Thrust (kN)', fontsize=11)
ax4.set_title('Thrust per Drone (Master Equation)', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# 5. Supra-saturation effectiveness
ax5 = axes[1, 1]
effectiveness = [c['effectiveness'] for c in drone_configs]
ax5.bar(range(NUM_DRONES), effectiveness, color=colors, edgecolor='black')
ax5.axhline(y=0.7, color='orange', linestyle='--', label='Good threshold')
ax5.axhline(y=1.0, color='green', linestyle='--', label='Maximum')
ax5.set_xlabel('Drone ID', fontsize=11)
ax5.set_ylabel('Effectiveness', fontsize=11)
ax5.set_title('Supra-Saturation Effectiveness', fontsize=12, fontweight='bold')
ax5.legend(fontsize=9)
ax5.grid(True, alpha=0.3, axis='y')

# 6. Θ_dilaton values
ax6 = axes[1, 2]
theta_vals = [c['theta'] for c in drone_configs]
ax6.bar(range(NUM_DRONES), theta_vals, color=colors, edgecolor='black')
ax6.set_xlabel('Drone ID', fontsize=11)
ax6.set_ylabel('Θ_dilaton', fontsize=11)
ax6.set_title('Dilaton Enhancement Factor', fontsize=12, fontweight='bold')
ax6.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))
ax6.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'RVG Unified Field Swarm Telemetry - {SCENARIO.replace("_", " ").title()}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 Telemetry Analysis Complete")

## 11. Material Performance Comparison

In [ ]:
# =============================================================================
# MATERIAL PERFORMANCE COMPARISON
# =============================================================================

if SCENARIO == 'material_comparison':
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Aggregate by material
    material_stats = {}
    for mat in MATERIALS.keys():
        mat_indices = [i for i, c in enumerate(drone_configs) if c['material'] == mat]
        if mat_indices:
            mat_alts = [final_positions[i][2] for i in mat_indices]
            mat_dists = [distances_traveled[i] if i < len(distances_traveled) else 0 for i in mat_indices]
            mat_thrust = np.mean([drone_configs[i]['thrust'] for i in mat_indices])
            mat_eff = np.mean([drone_configs[i]['effectiveness'] for i in mat_indices])
            
            material_stats[mat] = {
                'avg_alt': np.mean(mat_alts),
                'avg_dist': np.mean(mat_dists),
                'thrust': mat_thrust,
                'effectiveness': mat_eff,
                'B_sat': MATERIALS[mat]['B_sat'],
                'color': MATERIALS[mat]['color'][:3]
            }
    
    materials = list(material_stats.keys())
    
    # Plot 1: Thrust comparison
    thrusts = [material_stats[m]['thrust']/1000 for m in materials]
    colors = [material_stats[m]['color'] for m in materials]
    axes[0].bar(materials, thrusts, color=colors, edgecolor='black')
    axes[0].set_ylabel('Thrust (kN)', fontsize=11)
    axes[0].set_title('Thrust by Material', fontsize=12, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Average altitude achieved
    alts = [material_stats[m]['avg_alt'] for m in materials]
    axes[1].bar(materials, alts, color=colors, edgecolor='black')
    axes[1].set_ylabel('Avg Altitude (m)', fontsize=11)
    axes[1].set_title('Altitude by Material', fontsize=12, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Effectiveness vs B_sat
    B_sats = [material_stats[m]['B_sat'] for m in materials]
    effs = [material_stats[m]['effectiveness'] for m in materials]
    for i, mat in enumerate(materials):
        axes[2].scatter(B_sats[i], effs[i], s=200, 
                       color=material_stats[mat]['color'], 
                       edgecolors='black', label=mat)
    axes[2].set_xlabel('B_sat (T)', fontsize=11)
    axes[2].set_ylabel('Supra-Sat Effectiveness', fontsize=11)
    axes[2].set_title('Effectiveness vs Material Saturation', fontsize=12, fontweight='bold')
    axes[2].legend(fontsize=10)
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle('RVG Material Performance Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Material Rankings (by thrust):")
    sorted_mats = sorted(material_stats.items(), key=lambda x: x[1]['thrust'], reverse=True)
    for i, (mat, stats) in enumerate(sorted_mats, 1):
        print(f"  {i}. {mat}: {stats['thrust']/1000:.1f} kN, B_sat={stats['B_sat']}T, eff={stats['effectiveness']:.2f}")
else:
    print(f"Material comparison visualization only available in 'material_comparison' scenario.")
    print(f"Current scenario: {SCENARIO}")

## Summary

This notebook demonstrated multi-drone swarm simulation with **RVG Unified Field propulsion**:

### Key Findings:
- **Master Equation thrust** scales with Θ_dilaton(B) × ∇B² × Volume
- **MADA amplification** (200-500×) is critical for achieving high B_effective
- **Supra-saturation** (B >> B_sat) determines vacuum effect effectiveness
- **Minnealloy** (B_sat=2.85T) provides optimal performance per materials ranking

### Framework Integration:
- Dilaton enhancement Θ_dilaton(B) with 95 GeV resonance activation
- Material-specific B_sat for supra-saturation calculations
- Pulsing modulation (50-1000 Hz) for efficiency optimization

### References:
- [RVG Unified Field Theory](https://dx.doi.org/10.2139/ssrn.5381654) (Hofseth, 2025)
- [U.S. Patent #5,929,732 - MADA](https://patents.google.com/patent/US5929732A/en)

In [ ]:
print("\n" + "="*70)
print("RVG SWARM SIMULATION COMPLETE")
print("="*70)
print(f"\nScenario: {SCENARIO}")
print(f"Drones: {NUM_DRONES}")
print(f"Duration: {SIM_TIME}s")
print(f"\nNext steps:")
print("  1. Experiment with different scenarios (asymmetric, formation, mada_test)")
print("  2. Adjust MADA_K to test amplification effects")
print("  3. Compare material performance for your application")
print("  4. Integrate with ai/navigation.py for full 6DOF control")